# HVAC Association Graph

## Objective

Build and validate the HVAC association graph used by graph-constrained association rule mining (GC-ARM).

The graph construction, node inventory, derived-metric dependencies, and edge definitions are maintained in `src/graph.py`. This notebook imports that implementation so the notebook and the pipeline use one source of truth.

## 1. Project setup and imports

Locate the repository root, add it to Python's import path, and import the project modules.

In [ ]:
from pathlib import Path
import sys
import importlib
from collections import Counter

import networkx as nx
import matplotlib.pyplot as plt
import yaml

# Find the repository root whether this notebook is opened from the
# repository root or from the notebooks/ directory.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next(
    (
        candidate
        for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (candidate / "src" / "graph.py").exists()
        and (candidate / "config.yaml").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root containing src/graph.py and config.yaml."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. Dataset notes

Read the repository's dataset documentation so the graph analysis remains grounded in the project's variable definitions and HVAC system context.

In [ ]:
notes_path = PROJECT_ROOT / "data" / "dataset_notes.md"

if notes_path.exists():
    print(notes_path.read_text(encoding="utf-8"))
else:
    print(f"Dataset notes not found: {notes_path}")

## 3. HVAC system architecture

The graph represents the main HVAC equipment and water circuits:

- 3 chillers
- 3 cooling towers
- 3 primary chilled-water pumps
- 3 condenser-water pumps
- 2 secondary chilled-water pumps
- Primary chilled-water, secondary chilled-water, and condenser-water loops
- Outdoor-air conditions and control setpoints

Outdoor temperature can influence multiple variables through the control system. This is relevant when interpreting associations because it may act as a confounding factor.

## 4. Build the HVAC association graph

**Important:** Do not redefine the node inventory or edge lists in this notebook. `build_hvac_graph()` in `src/graph.py` is the canonical implementation used by the project pipeline.

In [ ]:
import src.graph as graph_module

importlib.reload(graph_module)

G = graph_module.build_hvac_graph()

print("Graph implementation:", graph_module.__file__)
print("Graph built successfully.")

## 5. Validate graph structure

Check node types, edge types, connectivity, and isolated nodes. The graph should contain 94 nodes, 140 edges, and one connected component after the reviewer-requested relationships have been added.

In [ ]:
node_type_counts = Counter(
    attributes["node_type"]
    for _, attributes in G.nodes(data=True)
)

edge_type_counts = Counter(
    attributes["edge_type"]
    for _, _, attributes in G.edges(data=True)
)

components = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True,
)

isolated_nodes = list(nx.isolates(G))

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

print("\nNode types:")
for node_type, count in sorted(node_type_counts.items()):
    print(f"{node_type}: {count}")

print("\nEdge types:")
for edge_type, count in sorted(edge_type_counts.items()):
    print(f"{edge_type}: {count}")

print("\nConnected components:", len(components))
print("Component sizes:", [len(component) for component in components])
print("Isolated nodes:", isolated_nodes)

assert G.number_of_nodes() == 94, "Unexpected number of graph nodes"
assert G.number_of_edges() == 140, "Unexpected number of graph edges"
assert len(components) == 1, "The HVAC graph must have exactly one connected component"
assert not isolated_nodes, "The HVAC graph must not contain isolated nodes"

print("\nGraph structure validation: PASS")

## 6. k-hop neighbourhood and allowed-pair counts

The neighbourhood and allowed-pair functions are imported from `src/graph.py`. Distances are treated as undirected. We report values for k = 1, 2, and 3 for comparison.

In [ ]:
test_node = "CHL_SW_TEMP_1"

for hop_count in [1, 2, 3]:
    neighbours = graph_module.k_hop_neighbourhood(
        G,
        test_node,
        hop_count,
    )
    print(
        f"k={hop_count}: {len(neighbours)} neighbours "
        f"from {test_node}"
    )

allowed_pairs_by_k = {
    hop_count: graph_module.allowed_pairs(G, hop_count)
    for hop_count in [1, 2, 3]
}

pair_counts = {
    hop_count: len(pairs)
    for hop_count, pairs in allowed_pairs_by_k.items()
}

print("\nAllowed variable-pair counts:")
for hop_count, count in pair_counts.items():
    print(f"k={hop_count}: {count}")

assert (
    pair_counts[1] <= pair_counts[2] <= pair_counts[3]
), "Allowed-pair counts should be non-decreasing as k increases"

print("\nAllowed-pair monotonicity: PASS")

## 7. Read the graph constraint from config.yaml

The pruning test must use the configured value of `graph.k`, rather than a hard-coded value.

In [ ]:
config_path = PROJECT_ROOT / "config.yaml"

with config_path.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

configured_k = config["graph"]["k"]

if isinstance(configured_k, bool) or not isinstance(configured_k, int):
    raise TypeError("config.yaml graph.k must be an integer")

print("Configured graph.k:", configured_k)

## 8. Test graph-constrained Apriori

Select two kinds of pairs from the actual HVAC graph:

1. A connected pair whose shortest path is greater than the configured k. Apriori should prune this pair.
2. A pair within the configured k-hop neighbourhood. Apriori should retain this pair.

The test uses repeated synthetic transactions so both individual items and their pair meet the support threshold.

In [ ]:
import src.gcarm as gcarm

importlib.reload(gcarm)

# Generate the allowed pairs using graph.k from config.yaml.
allowed = graph_module.allowed_pairs(G, configured_k)

# Find a connected pair whose shortest-path distance is greater than k.
far_pair = None
undirected_graph = G.to_undirected() if G.is_directed() else G

for source in sorted(undirected_graph.nodes):
    distances = nx.single_source_shortest_path_length(
        undirected_graph,
        source,
    )

    for target, distance in sorted(distances.items()):
        if source < target and distance > configured_k:
            far_pair = (source, target, distance)
            break

    if far_pair is not None:
        break

assert far_pair is not None, (
    f"No connected pair found at a distance greater than k={configured_k}"
)

far_u, far_v, far_distance = far_pair
far_itemset = frozenset({f"{far_u}=test", f"{far_v}=test"})

# Find a pair within the configured k-hop neighbourhood.
near_pair = min(
    (tuple(sorted(pair)) for pair in allowed),
    key=lambda pair: pair,
)

near_u, near_v = near_pair
near_itemset = frozenset({f"{near_u}=test", f"{near_v}=test"})

# Four identical transactions make the pair support equal to 1.0.
far_transactions = [
    {f"{far_u}=test", f"{far_v}=test"}
    for _ in range(4)
]

near_transactions = [
    {f"{near_u}=test", f"{near_v}=test"}
    for _ in range(4)
]

far_normal_results = gcarm.apriori(
    far_transactions,
    min_support=0.5,
    max_size=2,
)

far_constrained_results = gcarm.apriori(
    far_transactions,
    min_support=0.5,
    max_size=2,
    allowed_pairs=allowed,
)

near_constrained_results = gcarm.apriori(
    near_transactions,
    min_support=0.5,
    max_size=2,
    allowed_pairs=allowed,
)

# The far pair should exist without pruning and disappear with pruning.
assert far_itemset in far_normal_results, "Baseline Apriori did not find the distant pair"
assert far_itemset not in far_constrained_results, "Graph pruning did not reject the distant pair"

# The near pair should remain after graph pruning.
assert near_itemset in near_constrained_results, "Graph pruning incorrectly rejected a nearby pair"

print(f"Configured graph.k: {configured_k}")
print(
    f"Distant pair pruned ({far_distance} hops): "
    f"{far_u}, {far_v}"
)
print(
    f"Nearby pair retained (within {configured_k} hops): "
    f"{near_u}, {near_v}"
)
print("Graph-constrained Apriori integration test: PASS")

## 9. Visualize the HVAC association graph

The visualization is generated from the graph imported from `src/graph.py`. It does not reconstruct the graph or duplicate its edge definitions.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Use a simple graph for layout while retaining the typed MultiGraph
# for drawing each relationship category separately.
layout_graph = nx.Graph()
layout_graph.add_nodes_from(G.nodes(data=True))
layout_graph.add_edges_from((u, v) for u, v in G.edges())

positions = nx.spring_layout(
    layout_graph,
    seed=42,
    k=0.8,
    iterations=250,
)

sensor_nodes = [
    node
    for node, attributes in G.nodes(data=True)
    if attributes["node_type"] == "sensor_or_control"
]

metric_nodes = [
    node
    for node, attributes in G.nodes(data=True)
    if attributes["node_type"] == "derived_metric"
]

edge_styles = {
    "physical": {"color": "#4C78A8", "style": "solid"},
    "thermodynamic": {"color": "#E45756", "style": "dashed"},
    "control": {"color": "#54A24B", "style": "dotted"},
}

fig, ax = plt.subplots(figsize=(24, 18))

for edge_type, style in edge_styles.items():
    typed_edges = [
        (u, v)
        for u, v, attributes in G.edges(data=True)
        if attributes["edge_type"] == edge_type
    ]

    nx.draw_networkx_edges(
        layout_graph,
        positions,
        edgelist=typed_edges,
        edge_color=style["color"],
        style=style["style"],
        width=1.5,
        alpha=0.7,
        ax=ax,
    )

nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=sensor_nodes,
    node_color="#72A0C1",
    node_size=180,
    edgecolors="black",
    linewidths=0.5,
    ax=ax,
)

nx.draw_networkx_nodes(
    G,
    positions,
    nodelist=metric_nodes,
    node_color="#F2B45F",
    node_shape="s",
    node_size=260,
    edgecolors="black",
    linewidths=0.7,
    ax=ax,
)

nx.draw_networkx_labels(
    G,
    positions,
    font_size=8,
    ax=ax,
)

legend_items = [
    Patch(
        facecolor="#72A0C1",
        edgecolor="black",
        label="Sensor / Control Variable",
    ),
    Patch(
        facecolor="#F2B45F",
        edgecolor="black",
        label="Derived Metric",
    ),
    Line2D([0], [0], color="#4C78A8", linestyle="solid", label="Physical"),
    Line2D([0], [0], color="#E45756", linestyle="dashed", label="Thermodynamic"),
    Line2D([0], [0], color="#54A24B", linestyle="dotted", label="Control"),
]

ax.legend(
    handles=legend_items,
    loc="upper left",
    bbox_to_anchor=(1.01, 1),
    fontsize=11,
    frameon=True,
)

ax.set_title("HVAC Association Graph", fontsize=22, fontweight="bold")
ax.axis("off")
fig.tight_layout()

output_dir = PROJECT_ROOT / "notebooks" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

png_path = output_dir / "hvac_association_graph.png"
svg_path = output_dir / "hvac_association_graph.svg"

fig.savefig(png_path, dpi=400, bbox_inches="tight")
fig.savefig(svg_path, bbox_inches="tight")

plt.show()

print("PNG saved to:", png_path)
print("SVG saved to:", svg_path)

## 10. Final validation summary

This final cell provides a compact record of the graph and pruning checks performed in this notebook.

In [ ]:
print("=" * 48)
print("FINAL HVAC ASSOCIATION GRAPH VALIDATION")
print("=" * 48)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Connected components:", nx.number_connected_components(G))
print("Isolated nodes:", list(nx.isolates(G)))
print("Configured graph.k:", configured_k)

print("\nEdge counts:")
for edge_type, count in sorted(edge_type_counts.items()):
    print(f"{edge_type}: {count}")

print("\nAllowed variable pairs:")
for hop_count, count in pair_counts.items():
    print(f"k={hop_count}: {count}")

print("\nGraph validation: PASS")
print("Graph-constrained Apriori test: PASS")